![Egeria Logo](https://raw.githubusercontent.com/odpi/egeria/main/assets/img/ODPi_Egeria_Logo_color.png)

### Coco Pharmaceuticals Labs

----

# Mapping the solution components to the systems

[Erin Overview](https://egeria-project.org/practices/coco-pharmaceuticals/personas/erin-overview/) and [Peter Profile](https://egeria-project.org/practices/coco-pharmaceuticals/personas/peter-profile/) have laid out the solution components that implement Coco Pharmaceuticals' [strategic information supply chains](https://egeria-project.org/practices/coco-pharmaceuticals/scenarios/defining-information-supply-chains/overview/) — seventy-one of them, in eleven business system groups, joined by a hundred and thirty wires.  A solution component is a design element.  Before a supply chain can be *monitored* rather than merely described, each component has to be linked to the system that actually runs it.

That link is the `ImplementedBy` relationship, and this workbook creates it.

There are three estates to map against, and they are the point of the exercise:

| Estate | Systems | Where they came from |
|---|---|---|
| **Coco core** | 29 | [Gary Geeke's spreadsheet](https://egeria-project.org/practices/coco-pharmaceuticals/scenarios/cataloguing-infrastructure/overview/), loaded from `CocoComboArchive.omarchive` at startup |
| **Austin** | 45 | The acquired Austin site, loaded by [extending-the-systems-inventory](../extending-the-systems-inventory/README.md) |
| **Bucharest** | 30 | The acquired EKG site, loaded by the same notebook |

Erin and Peter took the components to Gary's inventory first and found it stopped where his responsibility stopped: the regulated layer of the business — laboratory, quality, batch release, safety — had no systems in it at all.  Gary asked the two acquisitions for their systems data to see how widespread the problem was, and it turned out not to be a problem at all: both acquisitions run a complete, modern stack for regulated manufacturing that the parent company has no equivalent of.

So the mapping does two jobs.  It creates the `ImplementedBy` links that connect design to reality, and — because a component that is implemented *only* at Austin or Bucharest is a component the original Coco Pharmaceuticals operation cannot run — it produces a report of exactly where the parent company's systems fall short.

----

In [ ]:
# All CSV files live in the data directory
from pathlib import Path

DATA_DIR = Path('./data').resolve()

print(f"Data directory: {DATA_DIR}")
for f in sorted(DATA_DIR.glob('*.csv')):
    print(f"  {f.name}")

In [ ]:
# Helper functions to load CSV files
import csv

def load_csv(filename):
    """Return a list of dicts from a CSV file in DATA_DIR."""
    path = DATA_DIR / filename
    with open(path, newline='', encoding='utf-8') as fh:
        return list(csv.DictReader(fh))

def preview(rows, n=3):
    """Print the first n rows and a row count."""
    print(f"{len(rows)} rows loaded")
    for row in rows[:n]:
        for k, v in row.items():
            print(f"  {k}: {v}")
        print()

In [ ]:
# Initialize pyegeria

import os
view_server = os.environ.get("EGERIA_VIEW_SERVER","qs-view-server")
url = os.environ.get("EGERIA_VIEW_SERVER_URL","https://host.docker.internal:9443")
user_id = "peterprofile"   # Peter can see every zone the systems sit in; Erin cannot see the manufacturing, compliance and depot zones
user_pwd = os.environ.get("EGERIA_USER_PASSWORD")
egeria_width = 150

print("\n")
print("Accessing view server " + view_server + " on platform " + url + " for user " + user_id)
print("\n")

# These packages support the different types of markdown display
from IPython.display import display, Markdown
from pyegeria import load_mermaid, render_mermaid
load_mermaid()

from datetime import datetime
from collections import defaultdict, Counter
import json
import time


# EgeriaTech combines many of the clients to call Egeria
from pyegeria import EgeriaTech

egeria_client = EgeriaTech(view_server, url, user_id, user_pwd)
token = egeria_client.create_egeria_bearer_token()

----

## The mapping

Two spreadsheets drive this workbook.

**`solution-components.csv`** lists the seventy-one fine-grained components from [strategic-supply-chain-analysis.md](../strategic-supply-chain-analysis.md) — their qualified name, business system group, component type, whether they are internal or external by design, and the supply chains they belong to.  The eleven business system groups themselves are containers rather than systems and are not mapped.

**`component-system-mapping.csv`** is the mapping itself: one row for every candidate system for every component, across all three estates.  It was built from the analysis in [strategic-supply-chain-system-matches.md](../strategic-supply-chain-system-matches.md), and each row carries a confidence:

| Confidence | Meaning | Linked by this workbook? |
|---|---|---|
| **Strong** | The system's description, or a loaded interaction between systems, names the function | Yes |
| **Probable** | The function plainly lives in this system, though nothing says so explicitly | Yes |
| **Possible** | The system *could* host it; the owner needs to confirm | No — reported, not linked |

The threshold is a variable, `LINK_CONFIDENCE`, so the Possible rows can be linked later without editing anything else.

----

In [ ]:
# Load the components and the mapping

components = load_csv('solution-components.csv')
mapping    = load_csv('component-system-mapping.csv')

print("Components:")
preview(components, 2)
print("Mapping rows:")
preview(mapping, 2)

In [ ]:
# What the mapping contains

LINK_CONFIDENCE = {"Strong", "Probable"}
ESTATES = ["Coco core", "Austin", "Bucharest"]

by_estate_conf = Counter((r['estate'], r['confidence']) for r in mapping)

lines = ["| Estate | Strong | Probable | Possible | Total |", "|---|---|---|---|---|"]
for e in ESTATES:
    s, p, q = (by_estate_conf[(e, c)] for c in ("Strong", "Probable", "Possible"))
    lines.append(f"| {e} | {s} | {p} | {q} | {s+p+q} |")
lines.append(f"| **All** | {sum(v for (e,c),v in by_estate_conf.items() if c=='Strong')} | "
             f"{sum(v for (e,c),v in by_estate_conf.items() if c=='Probable')} | "
             f"{sum(v for (e,c),v in by_estate_conf.items() if c=='Possible')} | {len(mapping)} |")
display(Markdown("\n".join(lines)))

internal = [c for c in components if c['scope'] == 'internal']
mapped   = {r['component_qualified_name'] for r in mapping}
print(f"{len(components)} components, {len(internal)} internal, {len(mapped)} with at least one candidate system, "
      f"{len([c for c in internal if c['component_qualified_name'] not in mapped])} internal components with none.")

----

## Creating the `ImplementedBy` relationships

`ImplementedBy` runs from the design element to the thing that implements it — here, from a solution component to a `SoftwareServer`.  It carries a small set of properties, and this workbook uses two of them:

* `role` is set to the **estate** the system belongs to, so that a component implemented in three places can be told apart by which place, and so the report below can be regenerated from Egeria rather than from the spreadsheet.
* `description` records the confidence and the note from the mapping, so anyone looking at the relationship later can see why it was drawn.

The workbook is idempotent.  For each component it first retrieves the implementations already linked in Egeria and skips any system that is already there, so it can be re-run after the mapping spreadsheet changes.  Nine of the archive's own components already carry `ImplementedBy` links (the ledgers, the two inventories, the two HazMat inventories, the sustainability servers); those are not in this mapping and are left untouched.

Every qualified name is resolved to a GUID before anything is created, and any name that does not resolve is reported rather than silently skipped — a component that is not in Egeria means the analysis file has not been loaded, and a system that is not in Egeria means the inventory notebook has not been run.

----

In [ ]:
# Resolve every qualified name in the mapping to a GUID

token = egeria_client.create_egeria_bearer_token()

guid_cache = {}
def guid_of(qualified_name):
    if qualified_name not in guid_cache:
        result = egeria_client.get_element_guid_by_unique_name(qualified_name)
        guid_cache[qualified_name] = None if result == "No elements found" else result
    return guid_cache[qualified_name]

unresolved_components = set()
unresolved_systems    = set()

for row in mapping:
    if guid_of(row['component_qualified_name']) is None:
        unresolved_components.add(row['component_qualified_name'])
    if guid_of(row['system_qualified_name']) is None:
        unresolved_systems.add(row['system_qualified_name'])

print(f"Resolved {len([g for g in guid_cache.values() if g])} of {len(guid_cache)} qualified names.")
if unresolved_components:
    print(f"\n{len(unresolved_components)} components not found - has strategic-supply-chain-analysis.md been loaded?")
    for q in sorted(unresolved_components): print("   ", q)
if unresolved_systems:
    print(f"\n{len(unresolved_systems)} systems not found - has extending-the-systems-inventory.ipynb been run?")
    for q in sorted(unresolved_systems): print("   ", q)

In [ ]:
# Create the ImplementedBy relationships for the linkable rows

def collect_guids(obj, found=None):
    """Walk a pyegeria response and gather every 'guid' value in it."""
    if found is None: found = set()
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k == 'guid' and isinstance(v, str): found.add(v)
            else: collect_guids(v, found)
    elif isinstance(obj, list):
        for v in obj: collect_guids(v, found)
    return found

def existing_implementations(component_guid):
    result = egeria_client.get_solution_component_implementations(component_guid)
    return set() if result == "No elements found" else collect_guids(result)

token = egeria_client.create_egeria_bearer_token()

created, skipped, deferred, failed = 0, 0, 0, 0
rows_by_component = defaultdict(list)
for row in mapping:
    rows_by_component[row['component_qualified_name']].append(row)

for component_qn, rows in rows_by_component.items():
    component_guid = guid_of(component_qn)
    if component_guid is None:
        continue
    already = existing_implementations(component_guid)

    for row in rows:
        if row['confidence'] not in LINK_CONFIDENCE:
            deferred += 1
            continue
        system_guid = guid_of(row['system_qualified_name'])
        if system_guid is None:
            failed += 1
            continue
        if system_guid in already:
            skipped += 1
            continue

        body = {
            "class": "NewRelationshipRequestBody",
            "properties": {
                "class": "ImplementedByProperties",
                "role": row['estate'],
                "description": f"{row['confidence']}: {row['note']}".rstrip(": ")
            }
        }
        egeria_client.link_design_to_implementation(design_desc_guid=component_guid,
                                                    implementation_guid=system_guid,
                                                    body=body)
        already.add(system_guid)
        created += 1
        print(f"Linked {row['component_name']}  ->  {row['system_name']}  [{row['estate']}, {row['confidence']}]")

print(f"\nDone - {created} relationships created, {skipped} already present, "
      f"{deferred} Possible rows deferred, {failed} rows with an unresolved system.")

----

## Checking one component in Egeria

Pick any component and ask Egeria what implements it.  The qualification and competency register is a good one to look at, because it is implemented in all three estates and read by processes in three different domains.

----

In [ ]:
# Retrieve the implementations of one component from Egeria

token = egeria_client.create_egeria_bearer_token()

component_qn = "CocoPharma::SolutionComponent::CompetencyRegister"
component_guid = guid_of(component_qn)

if component_guid:
    implementations = egeria_client.get_solution_component_implementations(component_guid)
    if implementations == "No elements found":
        print("No implementations linked yet.")
    else:
        print(json.dumps(implementations, indent=2)[:4000])
else:
    print(f"{component_qn} is not in Egeria.")

----

## The gap report

The report answers three questions, in order of how uncomfortable the answer is.

1. **Which components have no candidate system anywhere?**  These are capabilities the group does not have at all — nothing at the parent, nothing at either acquisition.
2. **Which components are outside the scope of the original Coco Pharmaceuticals operation?**  These have a Strong or Probable implementation at Austin or Bucharest but nothing at Coco core.  The group can run them; the parent cannot.
3. **Where is the parent's coverage unconfirmed?**  Components whose only Coco core candidate is Possible — the system might do it, and someone needs to find out.

It then shows the same picture by business system group and by supply chain, because a supply chain whose components are all outside the parent's scope is a supply chain the parent cannot yet monitor.

The report is displayed here and written to `system-mapping-report.md` alongside this notebook.

----

In [ ]:
# Build the gap report from the mapping

def strength(rows):
    """Best confidence in a set of rows: Strong > Probable > Possible > None."""
    order = ["Strong", "Probable", "Possible"]
    present = [c for c in order if any(r['confidence'] == c for r in rows)]
    return present[0] if present else None

comp_info = {c['component_qualified_name']: c for c in components}
per_estate = defaultdict(lambda: defaultdict(list))        # component -> estate -> rows
for r in mapping:
    per_estate[r['component_qualified_name']][r['estate']].append(r)

def classify(component_qn):
    est = per_estate.get(component_qn, {})
    core = strength(est.get("Coco core", []))
    acq  = strength(est.get("Austin", []) + est.get("Bucharest", []))
    if core in ("Strong", "Probable"):
        return "parent"                      # the original Coco operation can run it
    if core == "Possible":
        return "parent-unconfirmed"
    if acq in ("Strong", "Probable"):
        return "acquisitions-only"
    if acq == "Possible":
        return "acquisitions-possible"
    return "none"

LABEL = {"parent": "Parent covers it", "parent-unconfirmed": "Parent coverage unconfirmed",
         "acquisitions-only": "Acquisitions only", "acquisitions-possible": "Acquisitions only (Possible)",
         "none": "No system anywhere"}

classes = {c['component_qualified_name']: classify(c['component_qualified_name']) for c in internal}

def where(component_qn):
    est = per_estate.get(component_qn, {})
    parts = []
    for e in ESTATES:
        s = strength(est.get(e, []))
        if s: parts.append(f"{e} ({s})")
    return ", ".join(parts) if parts else "—"

out = []
out.append("# System mapping report\n")
out.append(f"Generated {datetime.now():%Y-%m-%d %H:%M} from `component-system-mapping.csv` — "
           f"{len(internal)} internal components, {len(mapping)} candidate rows, linked at confidence {sorted(LINK_CONFIDENCE)}.\n")

# ---- headline
counts = Counter(classes.values())
out.append("## Headline\n")
out.append("| Position | Components |")
out.append("|---|---|")
for key in ["parent", "parent-unconfirmed", "acquisitions-only", "acquisitions-possible", "none"]:
    out.append(f"| {LABEL[key]} | {counts.get(key, 0)} |")
out.append("")

# ---- 1. no system anywhere
out.append("## 1. No candidate system in any estate\n")
out.append("Capabilities the group does not have — nothing at the parent, nothing at either acquisition.\n")
out.append("| Component | Group | Supply chains |")
out.append("|---|---|---|")
for qn, cls in classes.items():
    if cls == "none":
        c = comp_info[qn]
        out.append(f"| {c['component_name']} | {c['business_system_group']} | {c['supply_chains']} |")
out.append("")

# ---- 2. outside the parent's scope
out.append("## 2. Outside the scope of the original Coco Pharmaceuticals operation\n")
out.append("Implemented at Austin and/or Bucharest, with no Strong or Probable system at Coco core.  "
           "The group can run these; the parent cannot.\n")
out.append("| Component | Group | Implemented at | Supply chains |")
out.append("|---|---|---|---|")
for qn, cls in classes.items():
    if cls in ("acquisitions-only", "acquisitions-possible"):
        c = comp_info[qn]
        out.append(f"| {c['component_name']} | {c['business_system_group']} | {where(qn)} | {c['supply_chains']} |")
out.append("")

# ---- 3. parent coverage unconfirmed
out.append("## 3. Parent coverage unconfirmed\n")
out.append("The only Coco core candidate is Possible — the system might do this, and the owner needs to say.\n")
out.append("| Component | Coco core candidate(s) | Elsewhere |")
out.append("|---|---|---|")
for qn, cls in classes.items():
    if cls == "parent-unconfirmed":
        c = comp_info[qn]
        core = ", ".join(sorted({r['system_name'] for r in per_estate[qn]["Coco core"]}))
        acq = ", ".join(f"{e} ({strength(per_estate[qn].get(e, []))})" for e in ("Austin", "Bucharest") if per_estate[qn].get(e))
        out.append(f"| {c['component_name']} | {core} | {acq or '—'} |")
out.append("")

# ---- by business system group
out.append("## By business system group\n")
out.append("| Group | Components | Parent covers | Unconfirmed | Acquisitions only | None |")
out.append("|---|---|---|---|---|---|")
groups = defaultdict(Counter)
for qn, cls in classes.items():
    groups[comp_info[qn]['business_system_group']][cls] += 1
for g in sorted(groups):
    k = groups[g]
    out.append(f"| {g} | {sum(k.values())} | {k['parent']} | {k['parent-unconfirmed']} | "
               f"{k['acquisitions-only'] + k['acquisitions-possible']} | {k['none']} |")
out.append("")

# ---- by supply chain
out.append("## By information supply chain\n")
out.append("A chain is only as monitorable at the parent as its least-covered component.\n")
out.append("| Supply chain | Components | Parent covers | Unconfirmed | Acquisitions only | None |")
out.append("|---|---|---|---|---|---|")
chains = defaultdict(Counter)
for qn, cls in classes.items():
    for isc in [s.strip() for s in comp_info[qn]['supply_chains'].split(';') if s.strip()]:
        chains[isc][cls] += 1
for isc in sorted(chains):
    k = chains[isc]
    out.append(f"| {isc} | {sum(k.values())} | {k['parent']} | {k['parent-unconfirmed']} | "
               f"{k['acquisitions-only'] + k['acquisitions-possible']} | {k['none']} |")
out.append("")

report = "\n".join(out)
display(Markdown(report))

report_path = Path('./system-mapping-report.md').resolve()
report_path.write_text(report, encoding='utf-8')
print(f"Report written to {report_path}")

----

## What the report says, and what comes next

The first table is short and the second is long, and that is the finding.  Almost nothing the strategic supply chains need is missing from the *group*; a great deal of it is missing from the *parent*.  The original Coco Pharmaceuticals operation grew its systems on a shoestring, one function at a time, and never bought a quality layer because nobody had asked it to prove anything.  The two acquisitions, bought independently in two jurisdictions, converged on the same regulated-manufacturing stack because inspection made them.  The integration question this raises is not *how do we absorb Austin* but *which of Austin's systems becomes the group standard*.

With the `ImplementedBy` links in place, each supply chain's implementation graph now reaches real systems.  The next step is the lineage beneath it: the interactions between the acquisitions' systems were loaded as `DataFlow` relationships by the inventory notebook, and because lineage relationships are multi-links, each strategic supply chain that passes over a hop can be given its own `DataFlow` between the same two systems, carrying that chain's qualified name.  That is what turns a designed supply chain into a monitored one.

----